# Lab 3: Qwen 2.5x1.5Berformance Optimization Challenge

## Objective
Optimize the training performance of a **Qwen 2.5x1.5B** model to achieve the highest possible **Model FLOPs Utilization (MFU)**.

## Workshop Structure
1. **Baseline profiling**: Profile an unoptimized training run
2. **Identify bottlenecks**: Use profiling tools to find inefficiencies
3. **Apply optimizations**: Implement kernel and parallelism optimizations
4. **Measure improvement**: Quantify MFU and throughput gains

## Topics Covered
- Distributed training parallelism (FSDP, EP, PP)
- Performance profiling with torch.profiler and Nsight
- Understanding arithmetic intensity
- Developing strategic optimization plans

## Grading (35 points)

| Component | Points | Description |
|-----------|--------|-------------|
| Profiling Analysis | 5 | Try profile with nsys file |
| MFU Improvement | 35 | Points based on achieved MFU (see rubric below) |

### MFU Scoring Rubric (35 points)
| MFU Achieved | Points |
|--------------|--------|
| ≥ 20.5% | 35 (full marks) |
| 1% - 20.5% | Linear: (MFU - 1) / 19.5 × 35 |
| < 1% | 0 |

## Hardware Requirements
- 8x NVIDIA H100 GPUs (or equivalent)
- Runtime container: `nvcr.io/nvidia/nemo-automodel:25.11`

## Submission
Submit the following files in your `submission/` folder:
1. **Nsight profile** (`automodel_profile_*.nsys-rep`) — from Part 2
2. **Optimized recipe** (`optimized_mixtral-Qwen 2.5x1.5B_benchmark.yaml`) — your configuration changes
3. **Benchmark results** (`benchmark_results.json`) — auto-generated MFU output

See **Part 5: Submission Checklist** for detailed instructions.

## Environment Setup

In [1]:
!nvidia-smi

Tue Mar 10 00:46:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!git clone https://github.com/NVIDIA-NeMo/Automodel.git
!mkdir -p submission
%cd Automodel
!git fetch origin zhiyul/llm-optimization-workshop
!git checkout zhiyul/llm-optimization-workshop

Cloning into 'Automodel'...
remote: Enumerating objects: 26046, done.
remote: Counting objects: 100% (528/528), done.
remote: Compressing objects: 100% (294/294), done.
remote: Total 26046 (delta 424), reused 241 (delta 234), pack-reused 25518 (from 3)
Receiving objects: 100% (26046/26046), 28.23 MiB | 27.56 MiB/s, done.
Resolving deltas: 100% (18726/18726), done.
/kaggle/working/Automodel
From https://github.com/NVIDIA-NeMo/Automodel
 * branch              zhiyul/llm-optimization-workshop -> FETCH_HEAD
Branch 'zhiyul/llm-optimization-workshop' set up to track remote branch 'zhiyul/llm-optimization-workshop' from 'origin'.
Switched to a new branch 'zhiyul/llm-optimization-workshop'


In [3]:
# install the nemo_automodel package
!pip install -e .

Obtaining file:///kaggle/working/Automodel
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 31.9 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.5/276.5 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 67.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.4/94.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 40.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 51.5 MB/s eta 0:00:00a 0:00:01

In [4]:
cd /kaggle/working/Automodel

/kaggle/working/Automodel


In [20]:
%%writefile ../submission/optimized_Qwen2.5-1.5B_benchmark.yaml
# Copyright (c) 2025, NVIDIA CORPORATION.  All rights reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# torchrun --nproc-per-node=8 --nnodes=1 nemo_automodel/recipes/llm/benchmark.py --config examples/llm_finetune/mistral/mixtral-8x7b-v0-1_benchmark.yaml
seed: 42

# Benchmark section
benchmark:
  warmup_steps: 5   
  peak_tflops: 65  # T4: 65, not H100: 989  
  nsys_start: -1    # Set to step number to profile (e.g., 10)
  nsys_end: -1      # Set to end step (e.g., 15)
  nsys_ranks: []    # e.g., [0] to profile rank 0
  num_nodes: 1

step_scheduler:
  global_batch_size: 256
  local_batch_size: 1   # open for change
  ckpt_every_steps: 50
  val_every_steps: 1000
  max_steps: 10

dist_env:
  backend: nccl
  timeout_minutes: 1

rng:
  _target_: nemo_automodel.components.training.rng.StatefulRNG
  seed: 1111
  ranked: true

model:
  _target_: nemo_automodel.NeMoAutoModelForCausalLM.from_pretrained
  pretrained_model_name_or_path: Qwen/Qwen2.5-1.5B
  torch_dtype: bf16
  force_hf: true

checkpoint:
  enabled: true
  checkpoint_dir: checkpoints/
  model_save_format: torch_save # torch_save or safetensors
  save_consolidated: false
  load_base_model: false

distributed:
  _target_: nemo_automodel.components.distributed.fsdp2.FSDP2Manager
  dp_size: None
  tp_size: 2      # 2 GPUs
  cp_size: 1
  use_hf_tp_plan: true

loss_fn:
  _target_: nemo_automodel.components.loss.masked_ce.MaskedCrossEntropy

# Use MockIterableDataset for benchmarking (faster, no I/O)
dataset:
  _target_: nemo_automodel.components.datasets.llm.mock_iterable_dataset.MockIterableDataset
  vocab_size: 100
  seq_len: 1024
  num_samples: 1000000

dataloader:
  _target_: torch.utils.data.DataLoader
  batch_size: null  # Dataset already yields batches
  # Note: model_config will be auto-injected by train_ft.py for PP models

optimizer:
  _target_: torch.optim.Adam
  betas: [0.9, 0.999]
  eps: 1e-8
  lr: 1.0e-5
  weight_decay: 0

lr_scheduler:
  lr_decay_style: cosine
  min_lr: 1.0e-6 

Writing ../submission/optimized_Qwen2.5-1.5B_benchmark.yaml


In [21]:
# Validate your config - run this before submitting!
import yaml
from typing import Any

# Fields that MUST remain unchanged
FROZEN_FIELDS = {
    "seed": 42,
    "benchmark.warmup_steps": 5,
    "benchmark.peak_tflops": 65,  # T4
    "benchmark.num_nodes": 1,
    "step_scheduler.global_batch_size": 256,
    "step_scheduler.max_steps": 10,
    "model._target_": "nemo_automodel.NeMoAutoModelForCausalLM.from_pretrained",
    "model.pretrained_model_name_or_path": "Qwen/Qwen2.5-1.5B",
    "model.torch_dtype": "bf16",
    "dataset._target_": "nemo_automodel.components.datasets.llm.mock_iterable_dataset.MockIterableDataset",
    "dataset.seq_len": 1024,
}

def get_nested_value(config: dict, key_path: str) -> Any:
    """Get nested value from config using dot notation."""
    keys = key_path.split(".")
    value = config
    for key in keys:
        if isinstance(value, dict) and key in value:
            value = value[key]
        else:
            return None
    return value

def validate_config(config_path: str) -> bool:
    """Validate that the config only changes allowed fields."""
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
    
    valid = True
    print("🔍 Validating config...")
    print("-" * 50)
    
    # Check frozen fields
    for field, expected in FROZEN_FIELDS.items():
        actual = get_nested_value(config, field)
        if actual != expected:
            print(f"❌ FROZEN field '{field}' was changed!")
            print(f"   Expected: {expected}, Got: {actual}")
            valid = False
    
    # Show distributed config
    dist = config.get("distributed", {})
    print(f"\n📊 Parallelism config:")
    print(f"   TP={dist.get('tp_size', 'None')}, "
          f"PP={dist.get('pp_size', 'None')}, "
          f"EP={dist.get('ep_size', 'None')}, "
          f"CP={dist.get('cp_size', 'None')}")
    print(f"   local_batch_size={config.get('step_scheduler', {}).get('local_batch_size', 1)}")
    
    print("-" * 50)
    if valid:
        print("✅ Config is valid! Ready to benchmark.")
    else:
        print("❌ Config validation FAILED. Please fix the errors above.")
    
    return valid

# Run validation
validate_config("../submission/optimized_Qwen2.5-1.5B_benchmark.yaml")

🔍 Validating config...
--------------------------------------------------

📊 Parallelism config:
   TP=2, PP=None, EP=None, CP=1
   local_batch_size=1
--------------------------------------------------
✅ Config is valid! Ready to benchmark.


True

---
## Part 1: Run baseline

> **Tip**: Running in a **terminal** is recommended for real-time output. Notebook cells also work, but may incur additional memory overhead.

The benchmark script will output:
- Per-iteration timing and loss
- Average MFU (Model FLOPs Utilization)
- JSON summary saved to `../submission/benchmark_results.json` (configured in YAML)

In [ ]:
%%bash
# Note: This is very slow with ~1% MFU - just run 2 steps for verification
# Also note that you need to run this in the Automodel directory
torchrun --nproc-per-node=8 --nnodes=1 \
    nemo_automodel/recipes/llm/benchmark.py \
    --config ../submission/optimized_mixtral-8x7b-v0-1_benchmark.yaml \
    --benchmark.warmup_steps=1 \
    --step_scheduler.max_steps=2

In [15]:
with open("/kaggle/working/Automodel/nemo_automodel/_transformers/auto_model.py", "r") as f:
    content = f.read()

old = '        attn_implementation: str = "flash_attention_2",'
new = '        attn_implementation: str = "sdpa",'

content = content.replace(old, new)

with open("/kaggle/working/Automodel/nemo_automodel/_transformers/auto_model.py", "w") as f:
    f.write(content)

print("Done!")

Done!


In [22]:
import subprocess, os

env = os.environ.copy()
env["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

process = subprocess.Popen(
    ["torchrun", "--nproc-per-node=2", "--nnodes=1",
     "--master-port=29500",
     "nemo_automodel/recipes/llm/benchmark.py",
     "--config", "../submission/optimized_Qwen2.5-1.5B_benchmark.yaml",
     "--benchmark.warmup_steps=1",
     "--step_scheduler.max_steps=2"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, cwd="/kaggle/working/Automodel", env=env
)

for line in process.stdout:
    print(line, end="")

process.wait()
print("Exit code:", process.returncode)

W0310 01:17:16.674000 629 torch/distributed/run.py:803] 
W0310 01:17:16.674000 629 torch/distributed/run.py:803] *****************************************
W0310 01:17:16.674000 629 torch/distributed/run.py:803] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0310 01:17:16.674000 629 torch/distributed/run.py:803] *****************************************
2026-03-10 01:17:24.032389: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-10 01:17:24.032408: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773105444.057890     635 cuda_dnn.cc:8579] Unabl

---
## Part 2: Profiling Analysis (5 points)

Use profiling tools to identify performance bottlenecks.

### Profiling Tools Available
1. **Nsight Systems**: GPU kernel analysis, timeline visualization
2. **torch.profiler**: PyTorch operation profiling
3. **CUDA Events**: Fine-grained kernel timing

We choose to use Nsight System inside the lab.

### Enable Nsight Profiling

Modify the benchmark config to enable Nsight profiling:

```yaml
benchmark:
  nsys_start: 1      # Start profiling at step 1 (after warmup)
  nsys_end: 1        # End profiling at step 1
  nsys_ranks: [0]    # Profile rank 0
```

### What to Look For
Open the `.nsys-rep` file in Nsight Systems GUI for analysis (from https://developer.nvidia.com/nsight-systems/get-started) and find bottlenecks.


### Running nsys with docker container
We need a specific version of nsys to work with Automodel, so here we use the docker container to get the right version of nsys and its dependencies. If you don't use the docker container, you are likely to run into errors with nsys.

In [ ]:
# step 1: run the following in a terminal to get docker
distribution=$(. /etc/os-release;echo $ID$VERSION_ID)
curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg
curl -fsSL https://nvidia.github.io/libnvidia-container/$distribution/libnvidia-container.list | \
  sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' | \
  sudo tee /etc/apt/sources.list.d/nvidia-container-toolkit.list

sudo apt update
sudo apt install -y nvidia-container-toolkit
sudo systemctl restart docker 

In [ ]:
# step 2: start the docker container and mount your local directory to the docker container
# run the following in a terminal
docker run --gpus all \
--ipc=host \
--ulimit memlock=-1 \
--ulimit stack=67108864 \
-p 8888:8888 \
-v /path/to/local/file:/docker_workspace/ \
-it nvcr.io/nvidia/nemo-automodel:25.11

In [ ]:
# step 3: run the following in the Automodel directory in the docker container
# cd /......./Automodel
export PYTHONPATH=$(pwd):/opt/Automodel:$PYTHONPATH

In [ ]:
# step 4: run the following nsys profiling command in the docker container
mkdir -p ../profile
TIMESTAMP=$(date +"%Y%m%d_%H%M%S")
PROFILE_FILE="../profile/automodel_profile_${TIMESTAMP}.nsys-rep"

nsys profile -s none \
    --trace=nvtx,cuda \
    --cudabacktrace=all \
    --cuda-graph-trace=node \
    --python-backtrace=cuda \
    --wait all \
    -o ${PROFILE_FILE} \
    --force-overwrite true \
    --capture-range=cudaProfilerApi \
    --capture-range-end=stop \
    torchrun --nproc-per-node=8 --nnodes=1 \
    nemo_automodel/recipes/llm/benchmark.py \
    --config ../submission/optimized_mixtral-8x7b-v0-1_benchmark.yaml \
    --benchmark.warmup_steps=1 \
    --step_scheduler.max_steps=2 \
    --benchmark.nsys_start=1 --benchmark.nsys_end=1 \
    --benchmark.nsys_ranks=[0]

---
## Part 3: Optimization Plan (35 points)

Based on your profiling analysis, develop an optimization strategy and update the recipe ../submission/optimized_mixtral-8x7b-v0-1_benchmark.yaml

### Hints for Available Optimization Techniques

#### 1. Distributed Parallelism
| Technique | Description | Config Key |
|-----------|-------------|------------|
| Tensor Parallel (TP) | Split model tensors across GPUs | `distributed.tp_size` |
| Expert Parallel (EP) | Distribute MoE experts across GPUs | `distributed.ep_size` |
| Pipeline Parallel (PP) | Split model layers across GPUs | `distributed.pp_size` |
| Data Parallel (FSDP) | Replicate model, split data | `distributed.dp_size` |
| Context Parallel (CP) | Split sequence across GPUs | `distributed.cp_size` |

#### 2. Batch Optimization
- **Local batch size**: Increase GPU utilization

#### 3. Memory Optimizations
- **activation_checkpointing**: Trade compute for memory

#### 4. Kernel
Go throught the code to see if there is any kernel option to try.

---
## Part 4: Implement Optimizations (35 points)

Create/update your optimized configuration and run the benchmark.

In [ ]:
%%bash
torchrun --nproc-per-node=8 --nnodes=1 \
    nemo_automodel/recipes/llm/benchmark.py \
    --config ../submission/optimized_mixtral-8x7b-v0-1_benchmark.yaml \
    --benchmark.json_output_path=../submission/benchmark_results.json

---
## Part 5: Submission Checklist

Before submitting, ensure your `submission/` folder contains all required files.

### Required Files

| File | Points | Description |
|------|--------|-------------|
| `automodel_profile_*.nsys-rep` | 5 | Nsight Systems profile from Part 2 |
| `optimized_mixtral-8x7b-v0-1_benchmark.yaml` | 35 | Your optimized recipe configuration |
| `benchmark_results.json` | — | Benchmark results with MFU (auto-generated) |

### Step-by-Step Submission Guide

**Step 1: Copy your Nsight profile to the submission folder**
```bash
cp ../profile/automodel_profile_*.nsys-rep ../submission/
```

**Step 2: Verify your optimized recipe**
> The recipe is already saved via the `%%writefile` cell in Environment Setup.  
> Re-run that cell if you made changes to your configuration.

**Step 3: Run the final benchmark to generate results**
```bash
torchrun --nproc-per-node=8 --nnodes=1 \
    nemo_automodel/recipes/llm/benchmark.py \
    --config ../submission/optimized_mixtral-8x7b-v0-1_benchmark.yaml
    --benchmark.json_output_path=../submission/benchmark_results.json
```

### Verify Your Submission

```bash
ls -la ../submission/
```

**Expected output:**
```
submission/
├── optimized_mixtral-8x7b-v0-1_benchmark.yaml  # Your optimized config
├── benchmark_results.json                       # Benchmark output with MFU score
└── automodel_profile_*.nsys-rep                 # Nsight profile for grading
```

### Grading Summary

| Component | Points | Criteria |
|-----------|--------|----------|
| Nsight Profile | 5 | Valid `.nsys-rep` file submitted |
| MFU Score | 35 | Based on `benchmark_results.json` (see rubric in intro) |
| **Total** | **40** | |